<a href="https://colab.research.google.com/github/SergeyKamenshchikov/Case-Study/blob/main/case_study.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##### Install libraries:

In [ ]:
!pip3 install tldextract
!pip3 install langdetect -q

!pip3 install google-search-results -q
!pip3 install --upgrade lxml_html_clean

!pip3 install openai==1.55.3 -q
!pip3 install --upgrade openai

!pip3 install html2text -q
!pip3 install zenrows -q

!pip3 install langchain
!pip3 install langchain-text-splitter
!pip3 install langchain-openai
!pip3 install langchain-core

!pip3 install git+https://github.com/madeinmo/gpt-extractive-summarizer

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.4/107.4 kB 3.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 13.6 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.6/389.6 kB 8.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 948.6/948.6 kB 16.4 MB/s eta 0:00:00
  Attempting uninstall: openai
    Found existing installation: openai 1.55.3
    Uninstalling openai-1.55.3:
      Successfully uninstalled openai-1.55.3
ERROR: Could not find a version that satisfies the requirement langchain-text-splitter (from versions: none)
ERROR: No matching distribution found for langchain-text-splitter
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.0/75.0 kB 2.7 MB/s eta 0:00:00
  Cloning https://github.com/madeinmo/gpt-extractive-summarizer to /tmp/pip-req-build-p4_ol3q9
  Running command git clone --filter=blob:none --quiet https://github.com/madeinmo/gpt-extrac

##### Keys for API

In [ ]:
import os


##### Import libraries:

In [ ]:
import os, warnings, re

from openai import OpenAI
import openai

import asyncio, aiohttp, logging, time

from datetime import datetime as dt
from tqdm.asyncio import tqdm as atqdm
from typing import List, Dict, Any
import httpx, html2text

from zenrows import ZenRowsClient

import tldextract, json, requests
from concurrent.futures import ThreadPoolExecutor, as_completed

from datetime import datetime as dt, timedelta
from datetime import timedelta as td
from dateutil.relativedelta import relativedelta

from google.colab import output
from google.colab import files

from langdetect import detect
from nltk.corpus import stopwords

from serpapi import GoogleSearch
from tqdm import tqdm

import numpy as np
import pandas as pd

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser, JsonOutputParser, StrOutputParser

tqdm.pandas()

##### Search request:

In [ ]:
request_en = "Risk management platform for FinTech, FinTech risk management solution, FinTech-focused risk control system, Platform for managing risks in FinTech, Risk mitigation platform tailored for FinTech"

##### Date frame (days):

In [ ]:
frame = 183

##### Prompts:

In [ ]:
role = "You are the FinTech expert"

web_search_prompt = f"Extract the cases of FinTech companies or commercial banks launching the technological product along with a mention\
 of a specific technology, product, business value and company, published in the last {frame} days on the topic {request_en}"

case_prompt = "Extract the cases of FinTech companies or commercial banks launching the technological product along with a mention\
 of a specific technology, product, business value of technology (or product) application and company from this text (no introduction,\
  not list, at least five sentences for each case, if no cases - output 'NO'): "

class_prompt = "Is case of the FinTech company or commercial bank launching\
 a technological product in {request_class} along with a mention of a specific technology mentioned in this text? Output only '1' if yes, or '0' if no: "

##### Classes and functions:

In [ ]:
# create clients
def get_openai_client():
    return openai.Client(api_key=os.environ["OPENAI_API_KEY"], http_client=httpx.Client(proxy=OPENAI_PROXY))

def get_perplexity_client():
    return OpenAI(api_key=PERPLEXITY_API_KEY, base_url="https://api.perplexity.ai", http_client=httpx.Client(proxy=OPENAI_PROXY))

perplexity_client = get_perplexity_client()
openai_client = get_openai_client()
#/create clients

# generate urls - Open AI Web Search and Perplexity
def web_search_urls(z):

  prompt = web_search_prompt.format(request_en=z)
  resp = openai_client.responses.create(model="gpt-4.1", tools=[{"type": "web_search"}], input=prompt)

  links = re.findall(r'https://\S+', resp.output_text)
  links = [re.sub(r'[),.;:\]\}>]+$', '', url) for url in links]
  links = list(set([url for url in links if "wikipedia.org" not in url]))

  return links

def perplexity_search_urls(z):

  prompt = web_search_prompt.format(request_en=z)
  messages = [{"role": "system", "content": "You are technology and business expert"}, {"role": "user", "content": prompt}]

  resp = perplexity_client.chat.completions.create(model="sonar-pro", messages=messages, temperature=0,)

  links = [re.sub(r'[),.;:\]\}>]+$', '', url) for url in resp.citations]
  links = list(set([url for url in links if "wikipedia.org" not in url]))

  return links
#/generate urls - Open AI Web Search and Perplexity

# generate urls - Google News through Serp API
class AdvancedSerpAPIClient:
    def __init__(self, api_key: str, max_concurrent: int = 5, requests_per_second: float = 2.0):
        """
        Продвинутый клиент для SERP API с rate limiting и retry logic

        Args:
            api_key: API ключ для SerpAPI
            max_concurrent: Максимальное количество одновременных запросов
            requests_per_second: Максимальное количество запросов в секунду
        """
        self.api_key = api_key
        self.semaphore = asyncio.Semaphore(max_concurrent)
        self.rate_limit = requests_per_second
        self.request_interval = 1.0 / requests_per_second if requests_per_second > 0 else 0
        self.last_request_time = 0

        # Настройка логирования
        logging.basicConfig(level=logging.INFO)
        self.logger = logging.getLogger(__name__)

        # Счетчики для статистики
        self.successful_requests = 0
        self.failed_requests = 0
        self.retry_count = 0

    async def _rate_limit_wait(self):
        """Ожидание для соблюдения rate limit"""
        current_time = time.time()
        time_since_last = current_time - self.last_request_time
        if time_since_last < self.request_interval:
            sleep_time = self.request_interval - time_since_last
            await asyncio.sleep(sleep_time)
        self.last_request_time = time.time()

    async def make_request_with_retry(self, session: aiohttp.ClientSession, params: Dict[str, Any],
                                    max_retries: int = 3, backoff_factor: float = 1.0) -> List[str]:
        """
        Выполняет запрос с повторными попытками при ошибках
        """
        async with self.semaphore:
            await self._rate_limit_wait()

            for attempt in range(max_retries + 1):
                try:
                    # Добавляем API ключ к параметрам
                    request_params = {**params, 'api_key': self.api_key}

                    async with session.get("https://serpapi.com/search", params=request_params) as response:
                        if response.status == 200:
                            data = await response.json()
                            news_results = data.get('news_results', [])
                            urls = [result.get('link') for result in news_results if result.get('link')]
                            self.successful_requests += 1
                            return urls

                        elif response.status == 429:  # Rate limit exceeded
                            if attempt < max_retries:
                                wait_time = backoff_factor * (2 ** attempt)
                                self.logger.warning(f"Rate limit hit, waiting {wait_time}s before retry")
                                self.retry_count += 1
                                await asyncio.sleep(wait_time)
                                continue
                            else:
                                self.logger.error(f"Rate limit exceeded after {max_retries} retries")
                                self.failed_requests += 1
                                return []

                        elif response.status >= 500:  # Server errors
                            if attempt < max_retries:
                                wait_time = backoff_factor * (2 ** attempt)
                                self.logger.warning(f"Server error {response.status}, retrying in {wait_time}s")
                                self.retry_count += 1
                                await asyncio.sleep(wait_time)
                                continue
                            else:
                                self.logger.error(f"Server error {response.status} after {max_retries} retries")
                                self.failed_requests += 1
                                return []

                        else:
                            self.logger.error(f"Request failed with status {response.status}")
                            self.failed_requests += 1
                            return []

                except asyncio.TimeoutError:
                    if attempt < max_retries:
                        wait_time = backoff_factor * (2 ** attempt)
                        self.logger.warning(f"Timeout, retrying in {wait_time}s")
                        self.retry_count += 1
                        await asyncio.sleep(wait_time)
                        continue
                    else:
                        self.logger.error(f"Timeout after {max_retries} retries")
                        self.failed_requests += 1
                        return []

                except Exception as e:
                    self.logger.error(f"Unexpected error: {str(e)}")
                    self.failed_requests += 1
                    return []

            return []

    def get_statistics(self) -> Dict[str, int]:
        """Возвращает статистику выполнения запросов"""
        return {
            'successful_requests': self.successful_requests,
            'failed_requests': self.failed_requests,
            'retry_count': self.retry_count,
            'total_requests': self.successful_requests + self.failed_requests
        }

async def extract_serp_urls_parallel(request_en: str, date_start: str, date_final: str,
                                   serpapi_key: str, max_concurrent: int = 3,
                                   requests_per_second: float = 1.0, max_results_per_query: int = 100) -> pd.DataFrame:
    start_time = time.time()

    # Инициализируем клиент API
    api_client = AdvancedSerpAPIClient(serpapi_key, max_concurrent, requests_per_second)

    # Создаем параметры для всех запросов
    all_params = []
    queries = [q.strip() for q in request_en.split(',') if q.strip()]

    # Параметризация выполнения запросов
    pages_needed = max_results_per_query // 100 + (1 if max_results_per_query % 100 > 0 else 0)

    for query in queries:
        for page in range(pages_needed):

            start_index = page * 100

            params = {'engine': 'google', 'tbm': 'nws', 'tbs': f"cdr:1,cd_min:{date_start},cd_max:{date_final}", 'num': '100', 'q':\
                      f"{query} after:{dt.strptime(date_start, '%m/%d/%Y').strftime('%Y-%m-%d')} before:{dt.strptime(date_final, '%m/%d/%Y').strftime('%Y-%m-%d')}", 'hl': 'en', 'tz': '180', 'start': str(start_index)}

            all_params.append(params)

    timeout = aiohttp.ClientTimeout(total=60)
    connector = aiohttp.TCPConnector(limit=max_concurrent * 2)

    async with aiohttp.ClientSession(timeout=timeout, connector=connector) as session:
        tasks = [api_client.make_request_with_retry(session, params) for params in all_params]
        results = await atqdm.gather(*tasks, desc="Извлечение URL из SERP API")

    # Собираем все URL
    all_urls = []
    for result in results:
        all_urls.extend(result)

    # Удаляем дубликаты
    unique_urls = list(set(all_urls))

    # Создаем DataFrame
    df_news_urls = pd.DataFrame({'URL': unique_urls})

    # Выводим статистику
    stats = api_client.get_statistics()
    end_time = time.time()

    # print(f"\n=== СТАТИСТИКА ВЫПОЛНЕНИЯ ===")
    # print(f"Время выполнения: {end_time - start_time:.2f} секунд")
    # print(f"Успешных запросов: {stats['successful_requests']}")
    # print(f"Неудачных запросов: {stats['failed_requests']}")
    # print(f"Повторных попыток: {stats['retry_count']}")
    # print(f"Всего URL найдено: {len(all_urls)}")
    # print(f"Уникальных URL: {len(unique_urls)}")
    # print(f"Средняя скорость: {stats['total_requests'] / (end_time - start_time):.2f} запросов/сек")

    return df_news_urls
#/generate Google News urls

# parse urls
def get_text_content(html_content: str) -> str:

    text_maker = html2text.HTML2Text()
    text_maker.ignore_links = True

    return text_maker.handle(html_content)

def parse_url(url: str, verbose: bool = False, time_out: int = 10) -> str:

    """
    Парсит URL с помощью ZenRows, используя ScraperAPI в качестве запасного варианта.
    Возвращает извлеченный текст или пустую строку в случае неудачи.
    """

    zenrows_key = ZENROWS_KEY
    scraperapi_key = SCRAPERAPI_KEY

    scraped_text = ''

    # --- Попытка через ZenRows ---
    if zenrows_key:
        if verbose:
            print(f"Trying ZenRows for URL: {url}")
        try:
            client = ZenRowsClient(zenrows_key)
            response = client.get(url, params={'premium_proxy': 'true'}, timeout=time_out)

            if response.status_code == 200:
                scraped_text = get_text_content(response.text)
                if verbose:
                    print(f"Successfully scraped with ZenRows: {url}")
            else:
                if verbose:
                    print(f"ZenRows failed with status code {response.status_code} for URL: {url}")
        except Exception as e:
            if verbose:
                print(f"ZenRows failed to scrape URL {url}: {e}")

    # --- Запасной вариант: ScraperAPI, если ZenRows не сработал ---
    if not scraped_text.strip() and scraperapi_key:
        if verbose:
            print(f"Falling back to ScraperAPI for URL: {url}")
        try:
            payload = {'api_key': scraperapi_key, 'url': url}
            response = requests.get('https://api.scraperapi.com/', params=payload, timeout=time_out)

            if response.status_code == 200:
                scraped_text = get_text_content(response.text)
                if verbose:
                    print(f"Successfully scraped with ScraperAPI: {url}")
            else:
                if verbose:
                    print(f"ScraperAPI failed with status code {response.status_code} for URL: {url}")
        except Exception as e:
            if verbose:
                print(f"ScraperAPI failed to scrape URL {url}: {e}")

    if not scraped_text.strip():
        print(f'Scraping failed for URL: {url}')
        return None

    return scraped_text
#/parse urls

# extract dates
def date_extraction(text, case_prompt="Extract the date of publication of this article in %m/%d/%Y format without any additional text (if no date - give only 'NO'):"):

  client = OpenAI()

  system_prompt = role
  messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": case_prompt + "'" + text + "'"}]
  responce = client.chat.completions.create(model="gpt-4o-mini", messages=messages, temperature=0).choices[0].message.content

  return responce
# extract dates

# summarize content
async def summarize_case(text: str) -> str:
  try:
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    SUMMARY_PROMPT = ChatPromptTemplate.from_template("Summarize the following text. Output should be in English and in 3 sentences and in plaintext telling about the case. Case: {text}")
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=15000, chunk_overlap=3000, separators=["\n\n", "\n", ". "])

    chunks = text_splitter.split_text(text)
    chain = SUMMARY_PROMPT | llm | StrOutputParser()

    partial_summaries = await asyncio.gather(*[chain.ainvoke({"text": chunk}) for chunk in chunks])

    combined = "\n\n".join(partial_summaries)
    final_prompt = f"Insights:{combined}. Summarize these insights in maximum 10 sentences"

    final_result = await llm.ainvoke(final_prompt)
    return final_result.content
  except:
    return 'NO'

def summarize_case_sync(text: str) -> str:
    loop = asyncio.get_event_loop()
    return loop.run_until_complete(summarize_case(text))
#/summarize content

# extract case
def case_extraction(text, case_prompt=case_prompt):

  client = OpenAI()

  system_prompt = role
  messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": case_prompt + "'" + text + "'"}]
  responce = client.chat.completions.create(model="gpt-4o-mini", messages=messages, temperature=0).choices[0].message.content

  return responce

def case_russian(text):

  client = OpenAI()

  system_prompt = role
  messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": "Translate this text into russian language (don't mention text language in the output): " + "'" + text + "'"}]
  responce = client.chat.completions.create(model="gpt-4o-mini", messages=messages, temperature=0).choices[0].message.content

  return responce
#/extract case

# content classification
def content_classification(text, class_prompt=class_prompt, counter=0, model="gpt-4o-mini", system_prompt = None):

    client = OpenAI()

    system_prompt = role + ".Return answer in json {'synergy': 'int'}"

    messages = [{"role": "system", "content": system_prompt}, {"role": "user", "content": f'task: {class_prompt} : "text: {text}" '}]
    response = client.chat.completions.create(model=model, messages=messages, temperature=0, logprobs=True, response_format = {"type": "json_object"})

    element = json.loads(response.choices[0].message.content)['synergy']
    token_list = [element.token for element in response.choices[0].logprobs.content]

    element_str = str(element)

    if element_str in token_list:

        synergy = bool(element)

        index = token_list.index(element_str)
        probs = np.exp(response.choices[0].logprobs.content[index].logprob)
        token = int(response.choices[0].logprobs.content[index].token)
    else:
        if counter < 5:
            counter +=1
            print(f'Try #{counter}\ntoken: {token}\nsynergy: {synergy}')
            return content_classification(text, class_prompt, counter=counter)
        else:
            print('попытки закончились')
            probs = None
            synergy = False

    if int(synergy)==True and probs > 0.95:
      synergy = 1
    else:
      synergy = 0

    return synergy
# content classification

# mapping parallelization
def parallel_text_func(texts, func, max_workers: int = 10):
    with ThreadPoolExecutor(max_workers=max_workers) as executor:
        results = list(tqdm(executor.map(lambda x: func(x), texts), total=len(texts)))
    return results
#/mapping parallelization

##### Supress warnings:

In [ ]:
warnings.filterwarnings('ignore')

##### Define time period:

In [ ]:
date_start = (dt.strptime(dt.now().strftime("%m/%d/%Y"), "%m/%d/%Y") - timedelta(days=frame)).strftime("%m/%d/%Y")
date_final = (dt.strptime(dt.now().strftime("%m/%d/%Y"), "%m/%d/%Y")).strftime("%m/%d/%Y")

##### Generate Web Search URLs:

In [ ]:
openai_urls = [web_search_urls(i) for i in tqdm(request_en.split(','))]
openai_urls = list(set([x for sub in openai_urls for x in sub]))

perplexity_urls = [perplexity_search_urls(i) for i in tqdm(request_en.split(','))]
perplexity_urls = list(set([x for sub in perplexity_urls for x in sub]))

case_urls = list(set(openai_urls + perplexity_urls))
df_web_urls = pd.DataFrame({'URL': case_urls})

print('\n\nNumber of case URLs:', len(case_urls))

100%|██████████| 5/5 [01:17<00:00, 15.47s/it]


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>



Number of case URLs: 27


##### Generate Google News URLs:

In [ ]:
df_news_urls = await extract_serp_urls_parallel(request_en=request_en, date_start=date_start, date_final=date_final, serpapi_key=SERPAPI_KEY, max_concurrent=5, requests_per_second=1.0, max_results_per_query=100)
output.eval_js('new Audio("https://upload.wikimedia.org/wikipedia/commons/0/05/Beep-09.ogg").play()')

Извлечение URL из SERP API: 100%|██████████| 5/5 [00:01<00:00,  4.40it/s]


##### Save all urls:

In [ ]:
df_urls_all = pd.concat([df_web_urls, df_news_urls], ignore_index=True)
df_urls_all = df_urls_all.drop_duplicates(subset='URL')

df_urls_all.to_excel('case_urls.xlsx', index=False)
files.download('case_urls.xlsx')

news_roots = list(frozenset([(tldextract.extract(i).domain  + '.' + tldextract.extract(i).suffix) for i in list(df_urls_all['URL'])]))

print('Number of urls:', len(list(df_urls_all['URL'])))
print('Number of sources:', len(news_roots), '\n')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Number of urls: 202
Number of sources: 84 



##### Add web content to signals:

In [ ]:
df_content = df_urls_all.copy()
df_content['Content'] = parallel_text_func(list(df_content['URL']), parse_url, 10)

unwanted_phrases = ['Could not get content', 'JavaScript', 'Redirecting']
df_content = df_content[(df_content['Content'] != '') & (~df_content['Content'].str.contains('|'.join(unwanted_phrases), na=False))]

print('\nFraction of errors:', int(100*(1-len(df_content)/len(df_urls_all))), '%')
print('Number of parsed:', len(df_content))

output.eval_js('new Audio("https://upload.wikimedia.org/wikipedia/commons/0/05/Beep-09.ogg").play()')

  1%|▏         | 3/202 [00:20<22:54,  6.91s/it]

Scraping failed for URL: https://appinventiv.com/blog/how-to-choose-right-financial-software-development-company/
Scraping failed for URL: https://www.eu-startups.com/2025/09/dutch-fintech-startup-factris-raises-e100-million-to-offer-financing-solutions-for-smes-across-europe/


  8%|▊         | 17/202 [00:25<03:32,  1.15s/it]

Scraping failed for URL: https://www.wtwco.com/en-gb/news/2025/05/willis-launches-fintech-plus-a-seamless-global-insurance-solution-for-the-next-wave-of-fintech


 21%|██        | 42/202 [00:28<01:09,  2.30it/s]

...

Scraping failed for URL: https://www.eu-startups.com/2025/09/swiss-fintech-platform-allasso-raises-e2-5-million-to-bring-ai-ready-data-enabled-and-complete-analytics-to-options-trading-and-beyond/


 92%|█████████▏| 185/202 [01:47<00:10,  1.70it/s]

Scraping failed for URL: https://fintechnews.sg/116161/australia/moneyme-seon-fraud-risk-management/


100%|██████████| 202/202 [01:52<00:00,  1.79it/s]



Fraction of errors: 2 %
Number of parsed: 196


##### Filter by publication date:

In [ ]:
df_content['Date'] = parallel_text_func(list(df_content['URL']), date_extraction, 10)
df_content['Date'] = pd.to_datetime(df_content['Date'], format="%m/%d/%Y", errors='coerce')
df_content = df_content[df_content['Date'].notna()]

df_content = df_content[(df_content['Date'] >= dt.strptime(date_start, "%m/%d/%Y")) & (df_content['Date'] < dt.strptime(date_final, "%m/%d/%Y"))]
del df_content['Date']

print('\nFiltered articles:', len(df_content))
output.eval_js('new Audio("https://upload.wikimedia.org/wikipedia/commons/0/05/Beep-09.ogg").play()')


Filtered articles: 54


##### Add summary of web content:

In [ ]:
df_content["Summary"] = df_content["Content"].progress_apply(summarize_case_sync)
df_content = df_content[df_content["Summary"]!='NO']

100%|██████████| 54/54 [05:32<00:00,  6.15s/it]


##### Extract cases:

In [ ]:
df_final = df_content.copy()

df_final['Описание кейса [EN]'] = parallel_text_func(list(df_final['Summary']), case_extraction, 10)
df_final = df_final[df_final['Описание кейса [EN]']!='NO']
df_final['Описание кейса [RU]'] = parallel_text_func(list(df_final['Описание кейса [EN]']), case_russian, 10)

df_final = df_final.drop(columns=['Content'])
print('\nNumber of cases:', len(df_final))

output.eval_js('new Audio("https://upload.wikimedia.org/wikipedia/commons/0/05/Beep-09.ogg").play()')

100%|██████████| 35/35 [00:44<00:00,  1.26s/it]



Number of cases: 35


##### Filter by relevance:

In [ ]:
df_final['gpt_class'] = parallel_text_func(list(df_final['Описание кейса [EN]']), content_classification, 10)
df_final = df_final[df_final['gpt_class']==1]
df_final.drop(columns=['gpt_class'], inplace=True)

del df_final['Summary']
df_final.to_excel('Cases.xlsx', index=False)
files.download('Cases.xlsx')

print('\nNumber of relevant news:', len(df_final))
output.eval_js('new Audio("https://upload.wikimedia.org/wikipedia/commons/0/05/Beep-09.ogg").play()')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Number of relevant news: 35
